# Discriminator routing test

Validates the proposed `BaseAction` hierarchy with a Pydantic discriminated union.

Three things are checked:

1. Pydantic correctly routes raw dicts to the right concrete class using the `type` field.
2. The generated JSON schema exposes all four variants (needed for `with_structured_output`).
3. The LLM reliably produces the correct discriminator value for mission prompts targeting each action type.


## 1. Imports and setup


In [ ]:
import json
from typing import Annotated
from typing import Literal
from typing import Union

from pydantic import BaseModel
from pydantic import Field

from laife.entities.utils.directions import CardinalDirection
from laife.params.load_env import load_env

load_env()
print("env loaded")

## 2. Define `BaseAction` and concrete action classes

This is the **proposed** shape — not yet in `action.py`. Each class carries a `type` literal that acts as the Pydantic discriminator.


In [ ]:
from enum import StrEnum


class ActionType(StrEnum):
    """Enum for action types, used as a discriminator in the BaseAction class."""

    MOVE = "move"
    BUILD = "build"
    CRAFT = "craft"
    PLAN = "plan"


class BaseAction(BaseModel):
    """Base class for all agent actions.

    `type` is intentionally NOT declared here. Each concrete subclass declares
    its own `type: Literal[ActionType.X]` field. Declaring it on the base as
    `ActionType` would cause a type-invariance error when subclasses narrow it to
    a Literal, since mutable class variable overrides must match exactly.
    """

    reason: str = Field(..., description="Why this action was chosen.")


class ActionMove(BaseAction):
    """Move in a direction."""

    # No default — forces the LLM to always emit the type field explicitly.
    type: Literal[ActionType.MOVE] = Field(ActionType.MOVE, description="Must be 'move'.")
    direction: CardinalDirection = Field(..., description="The direction to move.")
    distance: int = Field(..., description="Distance in units.")


class ActionBuild(BaseAction):
    """Build a structure."""

    type: Literal[ActionType.BUILD] = Field(ActionType.BUILD, description="Must be 'build'.")
    building_type: str = Field(..., description="Kind of building to construct.")
    description: str = Field(..., description="Description of the building.")
    size: int = Field(..., description="Size of the building.")


class ActionCraft(BaseAction):
    """Craft an item or utensil."""

    type: Literal[ActionType.CRAFT] = Field(ActionType.CRAFT, description="Must be 'craft'.")
    utensil_name: str = Field(..., description="Name of the item or utensil to craft.")
    description: str = Field(..., description="Description of the item or utensil.")


class ActionPlan(BaseAction):
    """Pause and plan the next steps."""

    type: Literal[ActionType.PLAN] = Field(ActionType.PLAN, description="Must be 'plan'.")


print("classes defined:", ActionMove, ActionBuild, ActionCraft, ActionPlan)


## 3. Build the discriminated union and inspect the JSON schema

The schema must list all four variants. This is what `with_structured_output` will send to the LLM.


In [ ]:
from pydantic import TypeAdapter

Actions = Annotated[
    ActionMove | ActionBuild | ActionCraft | ActionPlan,
    Field(discriminator="type"),
]

adapter = TypeAdapter(Actions)
schema = adapter.json_schema()
print(json.dumps(schema, indent=2))

## 4. Test Pydantic discriminator routing (no LLM)

Feed raw dicts with each `type` value and confirm the correct concrete class comes out.


In [ ]:
raw_cases = [
    (
        ActionType.MOVE,
        {
            "type": ActionType.MOVE,
            "reason": "explore",
            "direction": "north",
            "distance": 5,
        },
        ActionMove,
    ),
    (
        "build",
        {
            "type": "build",
            "reason": "need shelter",
            "building_type": "hut",
            "description": "small wood hut",
            "size": 3,
        },
        ActionBuild,
    ),
    (
        "craft",
        {
            "type": "craft",
            "reason": "need a tool",
            "utensil_name": "axe",
            "description": "stone axe",
        },
        ActionCraft,
    ),
    (
        "plan",
        {"type": "plan", "reason": "not sure what to do next"},
        ActionPlan,
    ),
]

all_ok = True
for label, raw, expected_cls in raw_cases:
    result = adapter.validate_python(raw)
    ok = isinstance(result, expected_cls)
    all_ok = all_ok and ok
    status = "✓" if ok else "✗"
    print(f"{status} type='{label}' → {type(result).__name__}  | {result}")

print()
print("ALL PASS" if all_ok else "SOME FAILED")

## 5. Wire up `ActionPicker` with the new `Actions` union

Uses `with_structured_output(Actions)` (the discriminated union) rather than the current `Action` wrapper.
Requires `OPENAI_API_KEY` in the environment (loaded above via `load_env()`).


In [ ]:
from dataclasses import dataclass
from dataclasses import field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate

from laife.llm_services.chat.config.chat_openai import ChatOpenAIConfig


class ActionEnvelope(BaseModel):
    """Envelope so the top-level schema is always type: 'object'.

    OpenAI's structured-output mode requires a top-level `type: "object"` schema.
    A discriminated union at the root produces `anyOf/oneOf`, which it rejects.
    Wrapping in this envelope keeps the root as an object and lets us use
    `method="function_calling"`, which has no schema-shape restrictions.
    """

    action: Actions = Field(..., discriminator="type", description="The chosen action.")


_PROMPT_TEMPLATE = (
    "You are an agent in a virtual world with a mission to complete.\n"
    "Available actions: move (travel in a direction), build (construct a building), "
    "craft (make an item or utensil), plan (think and plan next steps).\n\n"
    "Mission: {mission}\n\n"
    "Choose ONE action that best helps complete the mission."
)

action_prompt = ChatPromptTemplate([SystemMessagePromptTemplate.from_template(_PROMPT_TEMPLATE)])


@dataclass
class ActionPickerNew:
    """ActionPicker using the discriminated union of concrete action types."""

    chat_openai_config: ChatOpenAIConfig = field(default_factory=ChatOpenAIConfig)

    def __post_init__(self) -> None:
        """Initialize the ActionPickerNew instance."""
        model = self.chat_openai_config.create_chat_model()
        self.structured_llm = model.with_structured_output(
            ActionEnvelope, method="function_calling"
        )
        self.chain = action_prompt | self.structured_llm

    def invoke(self, mission: str) -> BaseAction:
        """Invoke the action picker and unwrap the envelope to the concrete action."""
        result = self.chain.invoke({"mission": mission})
        if not isinstance(result, ActionEnvelope):
            msg = f"Unexpected type: {type(result)}"
            raise TypeError(msg)
        return result.action


picker = ActionPickerNew()
print("ActionPickerNew ready")


## 6. Validate each action type end-to-end

Each prompt is designed to strongly imply a single action type. The test checks that the LLM produces the correct discriminator and that Pydantic routes it to the right class.


In [ ]:
llm_cases = [
    (
        "move",
        "Walk north for 10 units to reach the river.",
        ActionMove,
    ),
    (
        "build",
        "Construct a large wooden house at the current location.",
        ActionBuild,
    ),
    (
        "craft",
        "Make a stone axe so I can chop trees.",
        ActionCraft,
    ),
    (
        "plan",
        "I'm not sure where to go next. I need to think about the best strategy.",
        ActionPlan,
    ),
]

results = {}
all_ok = True

for label, mission, expected_cls in llm_cases:
    print(f"── {label} ──")
    try:
        result = picker.invoke(mission)
        ok = isinstance(result, expected_cls)
        all_ok = all_ok and ok
        status = "✓" if ok else f"✗ (got {type(result).__name__})"
        print(f"  status : {status}")
        # print(f"  type   : {result.type}")
        print(f"  reason : {result.reason}")
        print(f"  fields : {result.model_dump(exclude={'type', 'reason'})}")
        results[label] = result
    except Exception as exc:  # noqa: BLE001
        all_ok = False
        print(f"  ERROR: {exc}")
    print()

print("=" * 40)
print("ALL PASS" if all_ok else "SOME FAILED")